#### Einfaches RAG-System zur Ähnlichkeitssuche (mit JSON-Vektordatenbank)

Erorderlich: 

+ Installation von Ollama 
+ Starten des Ollama-Servers im Hintergrund
+ Download des konkret zu verwendenden Embedding-LLMs

In [6]:
from __future__ import annotations

import json
import math
from pathlib import Path

import ollama

MODEL = "qwen3-embedding:8b"
DB_PATH = Path("ollama_embeddings.json")


def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b)) # Bildung des Sklalarprodukts der beiden Vektoren
    norm_a = math.sqrt(sum(x * x for x in a)) # Berechnung der Länge des ersten Vektors
    norm_b = math.sqrt(sum(y * y for y in b)) # Berechnung der Länge des zweiten Vektors
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b) # Berechnung der Kosinus-Ähnlichkeit durch Division des Skalarprodukts durch das Produkt der Längen der beiden Vektoren


def embed_text(text: str) -> list[float]:
    # Aktuelle Ollama-Doku nutzt für Python die embed()-Funktion.
    response = ollama.embed(model=MODEL, input=text)

    # Je nach Rückgabeformat der Library robust behandeln:
    if "embeddings" in response:
        value = response["embeddings"]
        # Falls eine Liste mit genau einem Embedding zurückkommt
        if isinstance(value, list) and value and isinstance(value[0], list):
            return value[0]
        return value

    if "embedding" in response:
        return response["embedding"]

    raise ValueError(f"Unerwartetes Response-Format: {response}")

#ruft die Funktion embed_text auf, um die Embeddings für die gegebenen Texte zu erstellen, und speichert sie dann in einer JSON-Datei. Jede Eintragung in der Datei enthält den Originaltext und das zugehörige Embedding.
def build_index(texts: list[str], path: Path = DB_PATH) -> None:
    records = []
    for text in texts:
        records.append(
            {
                "text": text,
                "embedding": embed_text(text),
            }
        )

    with path.open("w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

# lädt die JSon-Datei
def load_index(path: Path = DB_PATH) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

# berechnet die Ähnlichkeit zwischen der Abfrage und den gespeicherten Embeddings, sortiert die Ergebnisse nach Ähnlichkeit und gibt die Top-K-Ergebnisse zurück.
def search(query: str, top_k: int = 3, path: Path = DB_PATH) -> list[dict]:
    query_embedding = embed_text(query)
    records = load_index(path)

    results = []
    for record in records:
        score = cosine_similarity(query_embedding, record["embedding"])
        results.append(
            {
                "text": record["text"],
                "score": score,
            }
        )

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]

In [7]:
texts = [
        "Die Vertragsfreiheit ist ein zentraler Grundsatz des Zivilrechts.",
        "Im Verwaltungsrecht spielt der Verhältnismäßigkeitsgrundsatz eine wichtige Rolle.",
        "Eine Anfechtung wegen Irrtums richtet sich nach den Regeln des BGB.",
        "Der Staat darf nur auf gesetzlicher Grundlage in Grundrechte eingreifen.",
        "Der Himmel ist blau und die Sonne scheint."
    ]
query = "Nach welchen Regeln kann ein Vertrag angefochten werden?"

In [9]:
build_index(texts)

hits = search(query, top_k=5)

print(f"\nQuery: {query}\n")
for i, hit in enumerate(hits, start=1):
    print(f"{i}. Score: {hit['score']:.4f}")
    print(f"   {hit['text']}")


Query: Nach welchen Regeln kann ein Vertrag angefochten werden?

1. Score: 0.6213
   Eine Anfechtung wegen Irrtums richtet sich nach den Regeln des BGB.
2. Score: 0.3825
   Die Vertragsfreiheit ist ein zentraler Grundsatz des Zivilrechts.
3. Score: 0.2843
   Der Staat darf nur auf gesetzlicher Grundlage in Grundrechte eingreifen.
4. Score: 0.2665
   Im Verwaltungsrecht spielt der Verhältnismäßigkeitsgrundsatz eine wichtige Rolle.
5. Score: 0.1736
   Der Himmel ist blau und die Sonne scheint.
